In [1]:
import torch
from torch import nn

In [2]:

import sentencepiece as spm

text = "Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence"
embedding = nn.Embedding(32000, 100)
# input_embedding(text)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
input_ids = tokenizer.encode(text, return_tensors="pt")

d:\anaconda3\envs\deeplearning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\anaconda3\envs\deeplearning\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
input_embedding = embedding(input_ids)

In [4]:
pos = torch.arange(0,31)
# pos = (pos.unsqueeze(1).repeat(1, 100))
# pos.shape
pos


tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30])

In [5]:
i = torch.arange(0,50)
d = torch.pow(10000, 2*i/100)
pos = torch.arange(0,31)
pos = (pos.unsqueeze(1).repeat(1, 50))

In [ ]:
arg = (pos / d)
sin_val = torch.sin(arg)
cos_val = torch.cos(arg)


In [7]:
stacked = torch.stack((sin_val, cos_val), dim=2)
stacked
final_pe = stacked.flatten(1)
final_pe

tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  ...,  1.0000e+00,
          0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  7.3912e-01,  ...,  1.0000e+00,
          1.2023e-04,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  9.9570e-01,  ...,  1.0000e+00,
          2.4045e-04,  1.0000e+00],
        ...,
        [ 2.7091e-01, -9.6261e-01, -9.6308e-01,  ...,  9.9999e-01,
          3.3663e-03,  9.9999e-01],
        [-6.6363e-01, -7.4806e-01, -8.4768e-01,  ...,  9.9999e-01,
          3.4866e-03,  9.9999e-01],
        [-9.8803e-01,  1.5425e-01, -1.7886e-01,  ...,  9.9999e-01,
          3.6068e-03,  9.9999e-01]])

In [8]:
x = input_embedding + final_pe 

In [11]:
wq = torch.rand(100, 100)
q = x @ wq 
q.shape

wk = torch.rand(100, 100)
k = x @ wk 
k.shape

wv = torch.rand(100, 100)
v = x @ wv 
v.shape

# ins = (q @ k.transpose(-2, -1))/100
# softmax = nn.Softmax(dim=-1)
# att = softmax(ins) @ v
# att.shape

torch.Size([1, 31, 100])

In [ ]:
# wq = torch.rand(31, 100)
# q = torch.rand(31, 100)
# k = torch.rand(31, 100)
# v = torch.rand(31, 100)
w = {}
for i in range(8):
    w[f'wiq {i}'] = torch.rand(100, 100)
    w[f'wik {i}'] = torch.rand(100, 100)
    w[f'wiv {i}'] = torch.rand(100, 100)

multihead = None
for i in range(8):
    qm = q @ w[f'wiq {i}']

    km = k @ w[f'wik {i}']

    vm = v @ w[f'wiv {i}']

    ins = (qm @ km.transpose(-2, -1))/100
    softmax = nn.Softmax(dim=-1)
    att = softmax(ins) @ vm
    if multihead == None:
        multihead = att
    else:
        print(multihead.shape)
        multihead = torch.cat((multihead, att), 2)
    print(att.shape)

torch.Size([1, 31, 100])
torch.Size([1, 31, 100])
torch.Size([1, 31, 100])
torch.Size([1, 31, 200])
torch.Size([1, 31, 100])
torch.Size([1, 31, 300])
torch.Size([1, 31, 100])
torch.Size([1, 31, 400])
torch.Size([1, 31, 100])
torch.Size([1, 31, 500])
torch.Size([1, 31, 100])
torch.Size([1, 31, 600])
torch.Size([1, 31, 100])
torch.Size([1, 31, 700])
torch.Size([1, 31, 100])


In [28]:
w[f'wo'] = torch.rand(8*100, 100)
multihead_final = multihead @ w[f'wo']

In [26]:
multihead.shape

torch.Size([1, 31, 800])

In [29]:
multihead_final.shape

torch.Size([1, 31, 100])

In [ ]:
class TransformerTranslation(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.w = nn.ParameterDict({})

        for i in range(6):
            for j in range(8):
                self.w[f'wiq_{i}_{j}'] = nn.Parameter(torch.randn(100, 100))
                self.w[f'wik_{i}_{j}'] = nn.Parameter(torch.randn(100, 100))
                self.w[f'wiv_{i}_{j}'] = nn.Parameter(torch.randn(100, 100))
                
            self.w[f'wq_{i}'] = nn.Parameter(torch.rand(100, 100))
            self.w[f'wk_{i}'] = nn.Parameter(torch.rand(100, 100))
            self.w[f'wv_{i}'] = nn.Parameter(torch.rand(100, 100))
            # h * dv, dm
            self.w[f'wo_{i}'] = nn.Parameter(torch.rand(8*100, 100))
            
        self.embedding = nn.Embedding(32000, 100)
        self.layernorm_1 = nn.LayerNorm(100)
        self.linear_1 = nn.Linear(100, 2048)
        self.relu_1 = nn.ReLU(2048)
        self.linear_2 = nn.Linear(2048, 100)
        self.layernorm_2 = nn.LayerNorm(100)
        # self.layernorm_1 = nn.Sequential(nn.Linear(1, 2), nn.Linear(2, 3))

    def forward(self, x):
        x = self.embedding(x)
        x_output = self.positional_embedding(x)

        for i in range(6):
            q = x_output @ self.w[f'wq_{i}']
            k = x_output @ self.w[f'wk_{i}']
            v = x_output @ self.w[f'wv_{i}']
            x = self.attention(q, k , v, i, self.w[f'wo_{i}'])
            print(f'x : {x.shape}')
            x_output = self.layernorm_1(x_output + x)
            print(f'x : {x.shape}')
            x = self.linear_1(x)
            print(f'x : {x.shape}')
            x = self.relu_1(x)
            print(f'x : {x.shape}')
            x = self.linear_2(x)
            print(f'x : {x.shape}')
            x = self.layernorm_2(x_output + x)
            print(f'x hasil akhir: {x.shape}')
            return x
    
    def positional_embedding(self, x):
        pos = torch.arange(0,31)
        i = torch.arange(0,50)
        d = torch.pow(10000, 2*i/100)
        pos = torch.arange(0,31)
        pos = (pos.unsqueeze(1).repeat(1, 50))
        arg = (pos / d)
        sin_val = torch.sin(arg)
        cos_val = torch.cos(arg)
        stacked = torch.stack((sin_val, cos_val), dim=2)
        stacked
        final_pe = stacked.flatten(1)
        final_pe
        x = input_embedding + final_pe 
        return x
    
    def attention(self, q, k, v, i, wo):
        multihead = None
        for j in range(8):
            qm = q @ self.w[f'wiq_{i}_{j}']

            km = k @ self.w[f'wik_{i}_{j}']

            vm = v @ self.w[f'wiv_{i}_{j}']

            ins = (qm @ km.transpose(-2, -1))/torch.sqrt(torch.tensor(100))
            softmax = nn.Softmax(dim=-1)
            att = softmax(ins) @ vm
            if multihead == None:
                multihead = att
            else:
                print(multihead.shape)
                multihead = torch.cat((multihead, att), 2)
            print(att.shape)
        multihead_final = multihead @ wo
        return multihead_final

In [ ]:
qm = torch.tensor([
    [[1., 0., 1., 0.],
     [0., 1., 0., 1.],
     [1., 1., 0., 0.]],

    [[0., 1., 1., 0.],
     [1., 0., 0., 1.],
     [0., 0., 1., 1.]]
])  # shape (2, 3, 4)

km = qm.clone()   # usually different, but OK for demo
vm = torch.tensor([
    [[1., 2., 3., 4.],
     [5., 6., 7., 8.],
     [9.,10.,11.,12.]],

    [[2., 3., 4., 5.],
     [6., 7., 8., 9.],
     [10.,11.,12.,13.]]
])  # shape (2, 3, 4)

mask = torch.tensor([
    [1, 1, 0],   # block last token for batch 0
    [1, 0, 0]    # only first token allowed for batch 1
])

scores = (qm @ km.transpose(-2, -1))/ 10.0
if mask != None:
    # print(f"scores shape : {scores.shape}")
    # print(f"Mask shape      : {mask.shape}")
    mask_expanded = mask.unsqueeze(1)  # shape [2, 1, 31] to broadcast over query dim
    scores = scores.masked_fill(mask_expanded == 0, float('-inf'))
softmax = nn.Softmax(dim=-1)
att = softmax(scores) @ vm



In [77]:
scores.shape[-1]

3

In [78]:

causal_mask = torch.triu(
    torch.ones(3, 3),
    diagonal=1
).bool()

In [79]:
causal_mask

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])

In [80]:
mask.shape

torch.Size([2, 3])

In [81]:
causal_mask.unsqueeze(1).shape

torch.Size([3, 1, 3])

In [82]:
scores.masked_fill(causal_mask == True, float('-inf'))

tensor([[[0.2000,   -inf,   -inf],
         [0.0000, 0.2000,   -inf],
         [0.1000, 0.1000, 0.2000]],

        [[0.2000,   -inf,   -inf],
         [0.0000, 0.2000,   -inf],
         [0.1000, 0.1000, 0.2000]]])

In [40]:
scores

tensor([[[0.2000, 0.0000,   -inf],
         [0.0000, 0.2000,   -inf],
         [0.1000, 0.1000,   -inf]],

        [[0.2000,   -inf,   -inf],
         [0.0000,   -inf,   -inf],
         [0.1000,   -inf,   -inf]]])

In [ ]:
M = torch.tensor([[1,2,3],[4,5,6],[7,8,9]])
out = torch.triu(torch.ones_like(M), diagonal=-1).bool()

In [ ]:
probs = softmax(logits, dim=-1)


In [111]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer

def positional_embedding(x):
        seq_len = x.shape[-1]
        pos = torch.arange(0,seq_len)
        i = torch.arange(0,50)
        d = torch.pow(10000, 2*i/100)
        pos = torch.arange(0,seq_len)
        pos = (pos.unsqueeze(1).repeat(1, 50))
        arg = (pos / d)
        sin_val = torch.sin(arg)
        cos_val = torch.cos(arg)
        stacked = torch.stack((sin_val, cos_val), dim=2)
        final_pe = stacked.flatten(1)
        return final_pe
    
class Attention(nn.Module):
    def __init__(self, dmodel, dk, mask_layer=False):
        super(Attention, self).__init__()
        # self.wiq = nn.Parameter(torch.randn(dmodel, dk))
        # self.wik = nn.Parameter(torch.randn(dmodel, dk))
        # self.wiv = nn.Parameter(torch.randn(dmodel, dk))
        self.mask_layer = mask_layer
        self.w = nn.ParameterDict({})
        # h*dv aslinya tapi aku anggep dk dv sama untuk sekarang
        self.wo = nn.Parameter(torch.empty(8*dk, dmodel))
        nn.init.xavier_uniform_(self.wo)
        for i in range(8):
            self.w[f'wiq_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
            nn.init.xavier_uniform_(self.w[f'wiq_{i}'])
            self.w[f'wik_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
            nn.init.xavier_uniform_(self.w[f'wik_{i}'])
            self.w[f'wiv_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
            nn.init.xavier_uniform_(self.w[f'wiv_{i}'])
    
    def forward(self, q, k, v, mask_padding=None):
        multihead = None
        for i in range(8):
            qm = q @ self.w[f'wiq_{i}'] 
            km = k @ self.w[f'wik_{i}'] 
            vm = v @ self.w[f'wiv_{i}'] 

            scores = (qm @ km.transpose(-2, -1))/ 10.0
            if mask_padding != None:
                # print(f"scores shape : {scores.shape}")
                # print(f"Mask shape      : {mask.shape}")
                mask_expanded = mask_padding.unsqueeze(1)  # shape [2, 1, 31] to broadcast over query dim
                scores = scores.masked_fill(mask_expanded == 0, float('-inf'))
            
            if self.mask_layer == True:
                h_scores = scores.shape[-1]
                
                causal_mask = torch.triu(
                    torch.ones(h_scores, h_scores),
                    diagonal=1
                ).bool().to(device)

                scores = scores.masked_fill(causal_mask == True, float('-inf'))
                
            softmax = nn.Softmax(dim=-1)
            att = softmax(scores) @ vm
            if multihead == None:
                multihead = att
            else:
                # print(multihead.shape)
                multihead = torch.cat((multihead, att), 2)
            # print(att.shape)
        multihead_final = multihead @ self.wo
        return multihead_final

        
    
class TransformerEncoder(nn.Module):
    def __init__(self):
        super(TransformerEncoder, self).__init__()
        self.w = nn.ParameterDict({})

        self.wq = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wq)
        self.wk = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wk)
        self.wv = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wv)
            
        self.attention_1 = Attention(100, 100)
        self.layernorm_1 = nn.LayerNorm(100)
        self.linear_1 = nn.Linear(100, 2048)
        self.relu_1 = nn.ReLU()
        self.linear_2 = nn.Linear(2048, 100)
        self.layernorm_2 = nn.LayerNorm(100)
        
    def forward(self, x_output, mask=None):
        # x = self.embedding(x)
        # x_output = self.positional_embedding(x)
        # q = x_output @ self.wq
        # k = x_output @ self.wk
        # v = x_output @ self.wv
        if mask == None:
            x = self.attention_1(x_output, x_output, x_output)
        else:
            x = self.attention_1(x_output, x_output, x_output, mask)
        # print(f'x : {x.shape}')
        x_output = self.layernorm_1(x_output + x)
        # print(f'x : {x.shape}')
        x = self.linear_1(x_output)
        # print(f'x : {x.shape}')
        x = self.relu_1(x)
        # print(f'x : {x.shape}')
        x = self.linear_2(x)
        # print(f'x : {x.shape}')
        x = self.layernorm_2(x_output + x)
        # print(f'x hasil akhir: {x.shape}')
        return x
    
class TransformerDecoder(nn.Module):
    def __init__(self):
        super(TransformerDecoder, self).__init__()
        self.w = nn.ParameterDict({})

        self.wq = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wq)
        self.wk = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wk)
        self.wv = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wv)
            
        self.attention_1 = Attention(100, 100, mask_layer=True)
        self.layernorm_1 = nn.LayerNorm(100)
        self.attention_2 = Attention(100, 100)
        self.layernorm_2 = nn.LayerNorm(100)
        self.linear_1 = nn.Linear(100, 2048)
        self.relu_1 = nn.ReLU()
        self.linear_2 = nn.Linear(2048, 100)
        self.layernorm_3 = nn.LayerNorm(100)
        
    def forward(self, x_output, k, v, mask=None):
        if mask == None:
            x = self.attention_1(x_output, x_output, x_output)
        else:
            x = self.attention_1(x_output, x_output, x_output, mask)
        # print(f'x : {x.shape}')
        x_output = self.layernorm_1(x_output + x)
        
        x = self.attention_2(x_output, k, v)
        x_output = self.layernorm_2(x_output + x)
        # print(f'x : {x.shape}')
        x = self.linear_1(x_output)
        # print(f'x : {x.shape}')
        x = self.relu_1(x)
        # print(f'x : {x.shape}')
        x = self.linear_2(x)
        x = self.layernorm_2(x_output + x)
        # print(f'x : {x.shape}')
        
        # print(f'x hasil akhir: {x.shape}')
        return x
    
class TransformerTranslation(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(32000, 100)
        # self.pos_embedding = nn.Embedding(50, 100)
        self.layers_encoder = nn.ModuleList([TransformerEncoder() for _ in range(6)])
        self.layers_decoder = nn.ModuleList([TransformerDecoder() for _ in range(6)])
        self.linear = nn.Linear(100, 3)
        self.softmax = nn.Softmax()

    def forward(self, x_enc, x_dec):
        batch_size, seq_len = x_dec.shape
        
        mask = torch.where((x_enc == 0), x_enc, 1)
        
        # Embeddings
        # print("x_enc (token ids)      :", x_enc.shape)
        x_enc_emb = self.embedding(x_enc)
        # print("x_enc_emb (embedding)  :", x_enc_emb.shape)
        pos_seq = x_enc.shape[1]
        x_enc_pos = positional_embedding(x_enc).unsqueeze(0)[:, :pos_seq, :]
        # print("x_enc_pos (pos emb)    :", x_enc_pos.shape)
        x_enc = x_enc_emb.to(device) + x_enc_pos.to(device)
        
        
       
        # print("x_enc (sum)            :", x_enc.shape)
        
        # print("x_dec (token ids)      :", x_dec.shape)
        x_dec_emb = self.embedding(x_dec)
        # print("x_dec_emb (embedding)  :", x_dec_emb.shape)
        # x_dec_pos = positional_embedding(x_dec).unsqueeze(0)
        pos_seq = x_dec.shape[1]
        x_dec_pos = positional_embedding(x_dec).unsqueeze(0)[:, :pos_seq, :]
        # print("x_dec_pos (pos emb)    :", x_dec_pos.shape)
        x_dec = x_dec_emb.to(device) + x_dec_pos.to(device)
        # ===== DEBUG PRINTS (DECODER) =====
        
        
        
        # print("x_dec (sum)            :", x_dec.shape)
        # print("x_dec device           :", x_dec.device)
        
        # x = self.embedding(x)
        # x = self.positional_embedding(x)
        for i in range(len(self.layers_encoder)):
            x_enc = self.layers_encoder[i](x_output=x_enc, mask=mask)
            x_dec = self.layers_decoder[i](x_output=x_dec, k=x_enc, v=x_enc)
        # print(f'x shape sebelum linear : {x.shape}')
        # x = torch.mean(x_dec, dim=1)
        # print(f'x shape sesudah mean  : {x.shape}')
        # x = self.linear(x)
        E = self.embedding.weight
        # print(f'x_dec : {x_dec.shape}')
        # print(f'E : {E.shape}')
        x = x_dec @ E.T
        # print(f'x : {x.shape}')
        # x = self.softmax(x)
        return x

    # def positional_embedding(self, input_embedding):
    #     pos = torch.arange(0,31)
    #     i = torch.arange(0,50)
    #     d = torch.pow(10000, 2*i/100)
    #     pos = torch.arange(0,31)
    #     pos = (pos.unsqueeze(1).repeat(1, 50))
    #     arg = (pos / d)
    #     sin_val = torch.sin(arg)
    #     cos_val = torch.cos(arg)
    #     stacked = torch.stack((sin_val, cos_val), dim=2)
    #     stacked
    #     final_pe = stacked.flatten(1)
    #     final_pe
    #     x = input_embedding.to(device) + final_pe.to(device)
    #     return x
    

    

model = TransformerTranslation().to(device)

# text = "Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence"
# text_indo = "Karena model kami tidak menggunakan rekursi maupun konvolusi, agar model dapat memanfaatkan urutan dari deretan (sequence)."

# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# input_ids = tokenizer.encode(text, return_tensors="pt")
# y_pred = model(input_ids)
# target = torch.randn_like(y_pred)
# loss = F.mse_loss(y_pred, target)

# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# optimizer.zero_grad()
# loss.backward()
# optimizer.step()


In [76]:
positional_embedding('s').unsqueeze(0)[:, :2, :].shape


torch.Size([1, 2, 100])

In [64]:
device = 'cuda'
models = TransformerTranslation().to(device)

# 2. Create Dummy Data
# Batch Size = 2, Sequence Length = 10, Vocab Size = 32000
batch_size = 2
seq_len = 16
vocab_size = 32000

# Random integers representing token IDs
src_data = torch.randint(1, vocab_size, (batch_size, seq_len)).to(device)
tgt_data = torch.randint(1, vocab_size, (batch_size, seq_len)).to(device) # In real life this is shifted right

# 3. Setup Optimizer and Loss
optimizer = torch.optim.Adam(models.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("Start Training Step...")

# 4. Training Step
optimizer.zero_grad()

# Forward pass
output = models(src_data, tgt_data) # [Batch, Seq, Vocab_Size]

# Reshape for CrossEntropyLoss
# Output: [Batch * Seq, Vocab_Size]
# Target: [Batch * Seq]
output_flat = output.view(-1, vocab_size)
tgt_flat = tgt_data.view(-1)

loss = criterion(output_flat, tgt_flat)
loss.backward()
optimizer.step()

print(f"Loss value: {loss.item()}")
print("Training step finished successfully.")

Start Training Step...


IndexError: too many indices for tensor of dimension 2

In [109]:
def test_translation(text, max_len=20):
    model.eval()
    model.to(device)

    # Encode source text
    src = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=16
    )["input_ids"].to(device)

    # Start token (<bos>)
    decoder_input = torch.tensor([[101]], device=device)

    generated_ids = []

    with torch.no_grad():
        for step in range(max_len):
            # Forward pass
            print(src.shape)
            output = model(src, decoder_input)
            # output: [1, seq_len, vocab]

            # Take last token logits
            last_token_logits = output[:, -1, :]

            # Greedy decoding
            predicted_id = torch.argmax(last_token_logits, dim=-1)[0].item()


            # Stop if <eos>
            if predicted_id == 102:
                break

            generated_ids.append(predicted_id)

            # Append predicted token
            decoder_input = torch.cat(
                [decoder_input,
                 torch.tensor([[predicted_id]], device=device)],
                dim=1
            )

    # Decode output tokens
    translated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return translated_text
translated_text = test_translation('i like to eat apple')



torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])
torch.Size([1, 16])


In [112]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
import random

# --- 1. SETUP & CONFIG ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Re-initialize the model using your class
model = TransformerTranslation().to(device)

# Tokenizer (Uncased has ~30k vocab, fits your Embedding(32000))
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# --- 2. DATA GENERATION & LOADING ---
def generate_synthetic_data(num_samples=1000):
    nouns = {"apple": "apel", "book": "buku", "car": "mobil", "cat": "kucing", "house": "rumah"}
    verbs = {"eats": "makan", "reads": "baca", "buys": "beli", "likes": "suka", "sees": "lihat"}
    subjects = {"i": "saya", "you": "kamu", "she": "dia", "we": "kami", "they": "mereka"}
    
    data = []
    for _ in range(num_samples):
        s_en, s_id = random.choice(list(subjects.items()))
        v_en, v_id = random.choice(list(verbs.items()))
        n_en, n_id = random.choice(list(nouns.items()))
        
        src = f"{s_en} {v_en} {n_en}"
        tgt = f"{s_id} {v_id} {n_id}"
        data.append((src, tgt))
    return data

train_data = generate_synthetic_data(1000)

class TranslationDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=16):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        src, tgt = self.data[idx]
        # Pad and truncate
        s_enc = self.tokenizer(src, max_length=self.max_len, padding='max_length', truncation=True, return_tensors="pt")
        with self.tokenizer.as_target_tokenizer():
            t_enc = self.tokenizer(tgt, max_length=self.max_len, padding='max_length', truncation=True, return_tensors="pt")
        return s_enc['input_ids'].squeeze(0), t_enc['input_ids'].squeeze(0)

dataset = TranslationDataset(train_data, tokenizer)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# --- 3. TRAINING LOOP ---
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss(ignore_index=0) 

print("Start Training...")
model.train()

epochs = 20
for epoch in range(epochs): 
    total_loss = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        
        # Shift Data for Teacher Forcing
        # Input to Decoder: Remove LAST token ([SEP]/Padding)
        decoder_input = tgt[:, :-1] 
        # Target Label: Remove FIRST token ([CLS])
        target_labels = tgt[:, 1:] 
        
        optimizer.zero_grad()
        
        # Forward Pass
        output = model(src, decoder_input) 
        
        # Calculate Loss
        # Flatten to [Batch * Seq, Vocab]
        loss = criterion(output.reshape(-1, 32000), target_labels.reshape(-1))
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    # Print average loss for the epoch
    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")

# --- 4. TESTING (YOUR FUNCTION) ---
print("\n--- Testing Translation ---")
model.eval()

def test_translation(text):
    src = tokenizer(text, return_tensors="pt", padding='max_length', max_length=16)['input_ids'].to(device)
    # Start with [CLS] (101)
    decoder_input = torch.tensor([[101]]).to(device) 
    
    print(f"Input: {text}")
    output_sentence = []
    
    for i in range(5): # Max 5 words
        with torch.no_grad():
            output = model(src, decoder_input) 
            # Look at the last token generated
            predicted_id = torch.argmax(output[:, -1, :], dim=-1).item()
            
            # Stop if [SEP] (102) or [PAD] (0)
            if predicted_id == 102 or predicted_id == 0:
                break
                
            word = tokenizer.decode([predicted_id])
            output_sentence.append(word)
            
            # Feed back into model
            decoder_input = torch.cat([decoder_input, torch.tensor([[predicted_id]]).to(device)], dim=1)
            
    print(f"Predicted: {' '.join(output_sentence)}")

# Run Tests
test_translation("i eats apple")
test_translation("she buys book")

Using device: cuda
Start Training...


d:\anaconda3\envs\deeplearning\Lib\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Epoch 1/20 | Loss: 14.0452
Epoch 2/20 | Loss: 5.3065
Epoch 3/20 | Loss: 1.4542
Epoch 4/20 | Loss: 0.2444
Epoch 5/20 | Loss: 0.0100
Epoch 6/20 | Loss: 0.0008
Epoch 7/20 | Loss: 0.0006
Epoch 8/20 | Loss: 0.0005
Epoch 9/20 | Loss: 0.0004
Epoch 10/20 | Loss: 0.0003
Epoch 11/20 | Loss: 0.0003
Epoch 12/20 | Loss: 0.0002
Epoch 13/20 | Loss: 0.0002
Epoch 14/20 | Loss: 0.0002
Epoch 15/20 | Loss: 0.0002
Epoch 16/20 | Loss: 0.0001
Epoch 17/20 | Loss: 0.0001
Epoch 18/20 | Loss: 0.0001
Epoch 19/20 | Loss: 0.0001
Epoch 20/20 | Loss: 0.0001

--- Testing Translation ---
Input: i eats apple
Predicted: say ##a ma ##kan ape
Input: she buys book
Predicted: dia bel ##i bu ##ku


In [113]:
def test_translation(text):
    src = tokenizer(text, return_tensors="pt", padding='max_length', max_length=16)['input_ids'].to(device)
    decoder_input = torch.tensor([[101]]).to(device) 
    
    print(f"Input: {text}")
    
    # Store the PREDICTED IDs here (integers), not the words
    predicted_ids = []
    
    for i in range(15): # Allow slightly longer generation
        with torch.no_grad():
            output = model(src, decoder_input) 
            predicted_id = torch.argmax(output[:, -1, :], dim=-1).item()
            
            if predicted_id == 102 or predicted_id == 0:
                break
            
            # Save the ID
            predicted_ids.append(predicted_id)
            
            # Feed back
            decoder_input = torch.cat([decoder_input, torch.tensor([[predicted_id]]).to(device)], dim=1)
            
    # Decode ALL at once (Removes ## automatically)
    final_sentence = tokenizer.decode(predicted_ids, skip_special_tokens=True)
    print(f"Predicted: {final_sentence}")

# Try again!
test_translation("i eats apple")
test_translation("she buys book")

Input: i eats apple
Predicted: saya makan apel
Input: she buys book
Predicted: dia beli buku


In [ ]:
from datasets import load_dataset

dataset = load_dataset("jw300", "en-id")
print(dataset)


In [56]:
decoder_input = torch.tensor([[101]]).to(device) 

with torch.no_grad():
    # Pad decoder input to match model's expected length if necessary, 
    # but your model handles variable length if pos_embedding is dynamic.
    # However, your model output shape is same as decoder input len.
    
    output = model(src, decoder_input) 
    print(output.shape)
    # Get last token probability
    last_token_logits = output[:, -1, :]
    predicted_id = torch.argmax(last_token_logits, dim=-1)[0].item()

    
    # Print what it picked
    word = tokenizer.decode([predicted_id])
    print(f"  Step {i+1}: {word}")
    
    # Append to decoder input for next step
    decoder_input = torch.cat([decoder_input, torch.tensor([[predicted_id]]).to(device)], dim=1)


x_enc (token ids)      : torch.Size([8, 16, 100])
x_enc_emb (embedding)  : torch.Size([8, 16, 100])
x_enc_pos (pos emb)    : torch.Size([1, 16, 100])
x_enc (sum)            : torch.Size([8, 16, 100])
x_dec : torch.Size([8, 16, 100])
E : torch.Size([32000, 100])
x : torch.Size([8, 16, 32000])
torch.Size([8, 16, 32000])


NameError: name 'i' is not defined

In [47]:
src

tensor([[  101,  1045,  5927,  2482,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  2057,  5927,  6207,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  2057,  5927,  6207,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  2027, 23311,  2482,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1045,  7777,  6207,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  2057, 23311,  2160,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  2017,  5927,  2160,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  2016,  9631,  6207,   102,     0,     0,     0,     0,     0,
    

In [39]:
src.shape

torch.Size([8, 16])

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
import pandas as pd
from sklearn.model_selection import train_test_split

# ==========================================
# 1. PPKM DATASET CLASS (Reads your CSV)
# ==========================================
class PPKMDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.tokenizer = tokenizer
        self.data = df
        
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Use .iloc to access rows by integer index
        row = self.data.iloc[idx]
        text = str(row['Tweet'])      # Your column name
        label = int(row['sentiment']) # Your column name (0, 1, 2)
        
        # Tokenize
        enc = self.tokenizer.encode_plus(
            text, 
            max_length=31, 
            padding='max_length', 
            truncation=True, 
            return_tensors='pt'
        )
        return enc['input_ids'].squeeze(0), torch.tensor(label)


# ==========================================
# 3. LOAD & SPLIT DATA
# ==========================================
# Read the file
filename = r"D:\download_d\archive\INA_TweetsPPKM_Labeled_Pure.csv"
try:
    print(f"Loading {filename}...")
    full_df = pd.read_csv(filename, sep='\t')
    print(f"Total rows: {len(full_df)}")
    
    # Split 80% Train, 20% Test
    train_df, test_df = train_test_split(full_df, test_size=0.2, random_state=42)
    print(f"Train size: {len(train_df)} | Test size: {len(test_df)}")
    
except Exception as e:
    print(f"Error loading file: {e}")
    # Fallback dummy data if file is missing
    train_df = pd.DataFrame({'Tweet': ['Dummy text'], 'sentiment': [1]})
    test_df = pd.DataFrame({'Tweet': ['Dummy text'], 'sentiment': [1]})

# Setup DataLoaders
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
train_dataset = PPKMDataset(train_df, tokenizer)
test_dataset = PPKMDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ==========================================
# 4. TRAINING LOOP
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

model = TransformerTranslation().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4) # Low LR for stability
criterion = nn.CrossEntropyLoss()

epochs = 5
print("\nStarting Training on PPKM Tweets...")

for epoch in range(1, epochs + 1):
    # --- TRAIN ---
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    for input_ids, labels in train_loader:
        input_ids, labels = input_ids.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    # --- TEST ---
    model.eval()
    test_correct = 0
    test_total = 0
    
    with torch.no_grad():
        for input_ids, labels in test_loader:
            input_ids, labels = input_ids.to(device), labels.to(device)
            outputs = model(input_ids)
            preds = torch.argmax(outputs, dim=1)
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    train_acc = 100 * correct / total
    test_acc = 100 * test_correct / test_total
    
    print(f"Epoch {epoch}: Loss = {train_loss/len(train_loader):.4f} | Train Acc = {train_acc:.2f}% | Val Acc = {test_acc:.2f}%")

Loading D:\download_d\archive\INA_TweetsPPKM_Labeled_Pure.csv...
Total rows: 23644
Train size: 18915 | Test size: 4729
Running on: cuda

Starting Training on PPKM Tweets...
Epoch 1: Loss = 0.6034 | Train Acc = 77.07% | Val Acc = 79.83%
Epoch 2: Loss = 0.5060 | Train Acc = 80.36% | Val Acc = 80.78%
Epoch 3: Loss = 0.4484 | Train Acc = 82.59% | Val Acc = 80.82%
Epoch 4: Loss = 0.3851 | Train Acc = 85.03% | Val Acc = 81.35%
Epoch 5: Loss = 0.3115 | Train Acc = 88.27% | Val Acc = 80.78%


In [97]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer

# ==========================================
# 1. DATASET
# ==========================================
class SimpleDataset(Dataset):
    def __init__(self):
        # 0 = Negative, 1 = Neutral, 2 = Positive
        self.data = [
            ("I love this movie", 2), ("Great film", 2), ("Awesome", 2), 
            ("I hate this", 0), ("Terrible", 0), ("Bad movie", 0), 
            ("It was okay", 1), ("Not bad", 1), ("Average", 1)
        ] * 20 # Duplicate data to make training loop last longer
        self.tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, label = self.data[idx]
        enc = self.tokenizer.encode_plus(
            text, max_length=31, padding='max_length', truncation=True, return_tensors='pt'
        )
        return enc['input_ids'].squeeze(0), torch.tensor(label)

# ==========================================
# 2. YOUR MODEL (CLEANED & FIXED)
# ==========================================
class Attention(nn.Module):
    def __init__(self, dmodel, dk):
        super(Attention, self).__init__()
        # self.wiq = nn.Parameter(torch.randn(dmodel, dk))
        # self.wik = nn.Parameter(torch.randn(dmodel, dk))
        # self.wiv = nn.Parameter(torch.randn(dmodel, dk))
        self.w = nn.ParameterDict({})
        # h*dv aslinya tapi aku anggep dk dv sama untuk sekarang
        self.wo = nn.Parameter(torch.empty(8*dk, dmodel))
        nn.init.xavier_uniform_(self.wo)
        for i in range(8):
            self.w[f'wiq_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
            nn.init.xavier_uniform_(self.w[f'wiq_{i}'])
            self.w[f'wik_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
            nn.init.xavier_uniform_(self.w[f'wik_{i}'])
            self.w[f'wiv_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
            nn.init.xavier_uniform_(self.w[f'wiv_{i}'])
    
    def forward(self, q, k, v):
        multihead = None
        for i in range(8):
            qm = q @ self.w[f'wiq_{i}'] 

            km = k @ self.w[f'wik_{i}'] 

            vm = v @ self.w[f'wiv_{i}'] 

            ins = (qm @ km.transpose(-2, -1))/ 10.0
            softmax = nn.Softmax(dim=-1)
            att = softmax(ins) @ vm
            if multihead == None:
                multihead = att
            else:
                # print(multihead.shape)
                multihead = torch.cat((multihead, att), 2)
            # print(att.shape)
        multihead_final = multihead @ self.wo
        return multihead_final

 
class Transformer(nn.Module):
    def __init__(self):
        super(Transformer, self).__init__()
        self.w = nn.ParameterDict({})

        self.wq = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wq)
        self.wk = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wk)
        self.wv = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wv)
            
        self.attention_1 = Attention(100, 100)
        self.layernorm_1 = nn.LayerNorm(100)
        self.linear_1 = nn.Linear(100, 2048)
        self.relu_1 = nn.ReLU()
        self.linear_2 = nn.Linear(2048, 100)
        self.layernorm_2 = nn.LayerNorm(100)
        
    def forward(self, x_output):
        # x = self.embedding(x)
        # x_output = self.positional_embedding(x)
        # q = x_output @ self.wq
        # k = x_output @ self.wk
        # v = x_output @ self.wv
        x = self.attention_1(x_output, x_output, x_output)
        # print(f'x : {x.shape}')
        x_output = self.layernorm_1(x_output + x)
        # print(f'x : {x.shape}')
        x = self.linear_1(x_output)
        # print(f'x : {x.shape}')
        x = self.relu_1(x)
        # print(f'x : {x.shape}')
        x = self.linear_2(x)
        # print(f'x : {x.shape}')
        x = self.layernorm_2(x_output + x)
        # print(f'x hasil akhir: {x.shape}')
        return x
    

# class TransformerTranslation(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.embedding = nn.Embedding(32000, 100)
#         self.pos_embedding = nn.Embedding(50, 100)
#         self.layers_input = nn.ModuleList([Transformer() for _ in range(2)]) # 2 Layers is enough
#         self.linear = nn.Linear(100, 3)

#     def forward(self, x):
#         batch_size, seq_len = x.shape
        
#         # Embeddings
#         x_emb = self.embedding(x)
#         positions = torch.arange(0, seq_len).expand(batch_size, seq_len).to(x.device)
#         x_pos = self.pos_embedding(positions)
#         x = x_emb + x_pos
        
#         # Layers
#         for layer in self.layers_input:
#             x = layer(x)
            
#         # Max Pooling (Best for classification)
#         x, _ = torch.max(x, dim=1)
        
#         # Output
#         x = self.linear(x)
#         return x
    
class TransformerTranslation(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(32000, 100)
        self.pos_embedding = nn.Embedding(50, 100)
        self.layers_input = nn.ModuleList([Transformer() for _ in range(2)])
        self.linear = nn.Linear(100, 3)
        self.softmax = nn.Softmax()

    def forward(self, x):
        batch_size, seq_len = x.shape
        
        # Embeddings
        x_emb = self.embedding(x)
        
        # Position Embeddings
        positions = torch.arange(0, seq_len).expand(batch_size, seq_len).to(x.device)
        x_pos = self.pos_embedding(positions)
        
        x = x_emb + x_pos
        
        # x = self.embedding(x)
        # x = self.positional_embedding(x)
        for layer in self.layers_input:
            x = layer(x)
        # print(f'x shape sebelum linear : {x.shape}')
        x, _ = torch.max(x, dim=1)
        # print(f'x shape sesudah mean  : {x.shape}')
        x = self.linear(x)
        # x = self.softmax(x)
        return x
    
    # ufdtid68======================================================
    
# class Attention(nn.Module):
#     def __init__(self, dmodel, dk):
#         super(Attention, self).__init__()
#         # self.wiq = nn.Parameter(torch.randn(dmodel, dk))
#         # self.wik = nn.Parameter(torch.randn(dmodel, dk))
#         # self.wiv = nn.Parameter(torch.randn(dmodel, dk))
#         self.w = nn.ParameterDict({})
#         # h*dv aslinya tapi aku anggep dk dv sama untuk sekarang
#         self.wo = nn.Parameter(torch.empty(8*dk, dmodel))
#         nn.init.xavier_uniform_(self.wo)
#         for i in range(8):
#             self.w[f'wiq_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
#             nn.init.xavier_uniform_(self.w[f'wiq_{i}'])
#             self.w[f'wik_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
#             nn.init.xavier_uniform_(self.w[f'wik_{i}'])
#             self.w[f'wiv_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
#             nn.init.xavier_uniform_(self.w[f'wiv_{i}'])
    
#     def forward(self, q, k, v):
#         multihead = None
#         for i in range(8):
#             qm = q @ self.w[f'wiq_{i}'] 

#             km = k @ self.w[f'wik_{i}'] 

#             vm = v @ self.w[f'wiv_{i}'] 

#             ins = (qm @ km.transpose(-2, -1))/ 10.0
#             softmax = nn.Softmax(dim=-1)
#             att = softmax(ins) @ vm
#             if multihead == None:
#                 multihead = att
#             else:
#                 # print(multihead.shape)
#                 multihead = torch.cat((multihead, att), 2)
#             # print(att.shape)
#         multihead_final = multihead @ self.wo
#         return multihead_final

        
    
# class Transformer(nn.Module):
#     def __init__(self):
#         super(Transformer, self).__init__()
#         self.w = nn.ParameterDict({})

#         self.wq = nn.Parameter(torch.empty(100, 100))
#         nn.init.xavier_uniform_(self.wq)
#         self.wk = nn.Parameter(torch.empty(100, 100))
#         nn.init.xavier_uniform_(self.wk)
#         self.wv = nn.Parameter(torch.empty(100, 100))
#         nn.init.xavier_uniform_(self.wv)
            
#         self.attention_1 = Attention(100, 100)
#         self.layernorm_1 = nn.LayerNorm(100)
#         self.linear_1 = nn.Linear(100, 2048)
#         self.relu_1 = nn.ReLU()
#         self.linear_2 = nn.Linear(2048, 100)
#         self.layernorm_2 = nn.LayerNorm(100)
        
#     def forward(self, x_output):
#         # x = self.embedding(x)
#         # x_output = self.positional_embedding(x)
#         # q = x_output @ self.wq
#         # k = x_output @ self.wk
#         # v = x_output @ self.wv
#         x = self.attention_1(x_output, x_output, x_output)
#         # print(f'x : {x.shape}')
#         x_output = self.layernorm_1(x_output + x)
#         # print(f'x : {x.shape}')
#         x = self.linear_1(x_output)
#         # print(f'x : {x.shape}')
#         x = self.relu_1(x)
#         # print(f'x : {x.shape}')
#         x = self.linear_2(x)
#         # print(f'x : {x.shape}')
#         x = self.layernorm_2(x_output + x)
#         # print(f'x hasil akhir: {x.shape}')
#         return x
    
# class TransformerTranslation(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.embedding = nn.Embedding(32000, 100)
#         self.pos_embedding = nn.Embedding(50, 100)
#         self.layers_input = nn.ModuleList([Transformer() for _ in range(6)])
#         self.linear = nn.Linear(100, 3)
#         self.softmax = nn.Softmax()

#     def forward(self, x):
#         batch_size, seq_len = x.shape
        
#         # Embeddings
#         x_emb = self.embedding(x)
        
#         # Position Embeddings
#         positions = torch.arange(0, seq_len).expand(batch_size, seq_len).to(x.device)
#         x_pos = self.pos_embedding(positions)
        
#         x = x_emb + x_pos
        
#         # x = self.embedding(x)
#         # x = self.positional_embedding(x)
#         for layer in self.layers_input:
#             x = layer(x)
#         # print(f'x shape sebelum linear : {x.shape}')
#         x, _ = torch.max(x, dim=1)
#         # print(f'x shape sesudah mean  : {x.shape}')
#         x = self.linear(x)
#         # x = self.softmax(x)
#         return x

# ==========================================
# 3. EXECUTION (This resets everything)
# ==========================================


# [Image of Transformer Encoder Architecture]


# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# Re-Initialize Model & Data
model = TransformerTranslation().to(device)
dataset = SimpleDataset()
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Re-Initialize Optimizer (Crucial Step!)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("Starting fresh training...")
for epoch in range(1, 16):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for input_ids, labels in dataloader:
        input_ids, labels = input_ids.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        predicted = torch.argmax(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch {epoch}: Loss = {total_loss/len(dataloader):.4f}, Accuracy = {acc:.2f}%")

Running on: cuda
Starting fresh training...
Epoch 1: Loss = 1.4034, Accuracy = 33.89%
Epoch 2: Loss = 1.1631, Accuracy = 28.33%
Epoch 3: Loss = 0.9150, Accuracy = 53.89%
Epoch 4: Loss = 0.4829, Accuracy = 81.67%
Epoch 5: Loss = 0.0449, Accuracy = 100.00%
Epoch 6: Loss = 0.0041, Accuracy = 100.00%
Epoch 7: Loss = 0.0029, Accuracy = 100.00%
Epoch 8: Loss = 0.0024, Accuracy = 100.00%
Epoch 9: Loss = 0.0021, Accuracy = 100.00%
Epoch 10: Loss = 0.0019, Accuracy = 100.00%
Epoch 11: Loss = 0.0017, Accuracy = 100.00%
Epoch 12: Loss = 0.0015, Accuracy = 100.00%
Epoch 13: Loss = 0.0014, Accuracy = 100.00%
Epoch 14: Loss = 0.0013, Accuracy = 100.00%
Epoch 15: Loss = 0.0012, Accuracy = 100.00%


In [197]:
from torch.utils.data import Dataset, DataLoader
import torch

class ToySentimentDataset(Dataset):
    def __init__(self, tokenizer):
        # 0 = Negative, 1 = Neutral, 2 = Positive
        self.tokenizer = tokenizer
        
        # YOUR 50 SENTENCES (from before)
        self.sentences = [
            "I love this movie so much", "What a horrible film", "It was okay, not the best", 
            "Absolutely fantastic experience", "Terrible plot and bad acting", "Mediocre and boring",
            "I enjoyed every moment", "I hate every part of it", "Not good, not bad",
            "It was great and fun", "Awful and disappointing", "Satisfying but could be better",
            "Loved the characters and story", "Worst movie ever", "Quite average film",
            "Amazing visuals, great soundtrack", "Bad direction ruined it", "Neutral feelings about this",
            "I really liked it", "I don't dislike it", "It is not my type of movie",
            "Wonderful and heartwarming", "Terrible ending", "Nothing special",
            "Fantastic pace and acting", "Poorly written script", "I feel indifferent",
            "Wonderful, I recommend it", "Not worth watching", "Decent enough",
            "Excellent movie overall", "It lacked depth", "Good but not perfect",
            "Too many flaws", "Pretty fun to watch", "Would not watch again",
            "I’m on the fence", "Brilliant performance", "Disappointing experience",
            "Balanced — some good, some bad", "Loved some parts, hated others",
            "Fine for a relaxed evening", "Terrible from start to finish", "Nothing memorable",
            "Awesome cinematic journey", "Mediocre acting", "Not bad at all",
            "You should watch it", "I don’t recommend this movie", "Neutral review"
        ]

        self.labels = [
            2,0,1,2,0,1,2,0,1,2,
            0,1,2,0,1,2,0,1,2,1,
            1,2,0,1,2,0,1,2,0,1,
            2,0,1,2,0,2,0,1,2,0,
            1,0,1,2,0,1,2,1,2,1
        ]

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        # We handle tokenization here to keep the loop clean
        enc = self.tokenizer.encode_plus(
            self.sentences[idx],
            max_length=31,            # Fixed length like your manual code expected
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return enc['input_ids'].squeeze(0), torch.tensor(self.labels[idx])

In [198]:
# 1. Setup Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. Initialize the dataset using the class above
# CHANGE: Use ToySentimentDataset instead of SimpleDataset
dataset = ToySentimentDataset(tokenizer) 
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# 3. Setup Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerTranslation().to(device) # Make sure class is defined previously

# 4. Setup Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print(f"Training on {len(dataset)} sentences...")

# 5. Run Training
for epoch in range(1, 31): # 30 Epochs to be safe
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for input_ids, labels in dataloader:
        input_ids, labels = input_ids.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        predicted = torch.argmax(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch}: Loss = {total_loss/len(dataloader):.4f}, Accuracy = {acc:.2f}%")

Training on 50 sentences...
Epoch 1: Loss = 1.6339, Accuracy = 38.00%
Epoch 5: Loss = 1.1209, Accuracy = 30.00%
Epoch 10: Loss = 0.4073, Accuracy = 76.00%
Epoch 15: Loss = 0.2911, Accuracy = 86.00%
Epoch 20: Loss = 0.0462, Accuracy = 100.00%
Epoch 25: Loss = 0.0115, Accuracy = 100.00%
Epoch 30: Loss = 0.0064, Accuracy = 100.00%
